# Massachusetts Roads — prepare 512x512 tiles (Colab)

Run once. No GPU needed. Downloads the full dataset (~8 GB) from the original
UofT mirror, tiles it, and stores `tiles.zip` in your Google Drive under
`MyDrive/roadx/` (~4 GB) so training sessions can reuse it.

*Runtime -> Run all*, approve the Drive permission popup, then wait (~30-45 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/roadx


In [ ]:
import os; os.makedirs('roadx/data', exist_ok=True); os.makedirs('configs', exist_ok=True); open('roadx/__init__.py','w').close(); open('roadx/data/__init__.py','w').close()


In [ ]:
%%writefile roadx/data/download.py
"""Download the Massachusetts Roads Dataset from the original UofT mirror.

Images are 1500x1500 aerial tiles (.tiff) with matching binary road masks (.tif).
"""

import argparse
import re
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

BASE = "https://www.cs.toronto.edu/~vmnih/data/mass_roads"
SPLITS = ("train", "valid", "test")


def list_files(split: str, kind: str) -> list[str]:
    """Return download URLs listed in the split's index page. kind: 'sat' or 'map'."""
    url = f"{BASE}/{split}/{kind}/index.html"
    with urllib.request.urlopen(url, timeout=30) as r:
        html = r.read().decode("utf-8", errors="ignore")
    hrefs = re.findall(r'href="([^"]+\.tiff?)"', html)
    out = []
    for h in hrefs:
        if h.startswith("http"):
            out.append(h)
        else:
            out.append(f"{BASE}/{split}/{kind}/{h.lstrip('./')}")
    return sorted(set(out))


def fetch(url: str, dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    return dest


def download_split(split: str, out: Path, n: int | None, workers: int = 8) -> None:
    sat_urls = {Path(u).stem: u for u in list_files(split, "sat")}
    map_urls = {Path(u).stem: u for u in list_files(split, "map")}
    stems = sorted(set(sat_urls) & set(map_urls))
    if n is not None:
        stems = stems[:n]
    print(f"[{split}] {len(stems)} image/mask pairs")

    (out / split / "sat").mkdir(parents=True, exist_ok=True)
    (out / split / "map").mkdir(parents=True, exist_ok=True)

    jobs = []
    for s in stems:
        jobs.append((sat_urls[s], out / split / "sat" / Path(sat_urls[s]).name))
        jobs.append((map_urls[s], out / split / "map" / Path(map_urls[s]).name))

    done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fetch, u, d): d for u, d in jobs}
        for f in as_completed(futs):
            f.result()
            done += 1
            if done % 20 == 0 or done == len(jobs):
                print(f"[{split}] {done}/{len(jobs)} files")


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--out", type=Path, default=Path("data/raw"))
    p.add_argument("--train-n", type=int, default=50, help="train pairs to fetch")
    p.add_argument("--valid-n", type=int, default=14)
    p.add_argument("--test-n", type=int, default=10)
    p.add_argument("--all", action="store_true", help="download the full dataset")
    args = p.parse_args()

    counts = {"train": args.train_n, "valid": args.valid_n, "test": args.test_n}
    for split in SPLITS:
        download_split(split, args.out, None if args.all else counts[split])
    print("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile roadx/data/tile.py
"""Cut 1500x1500 raw images and masks into 512x512 training patches.

Tiles containing a large fraction of blank (white, no-data) pixels are dropped —
the Massachusetts dataset pads incomplete tiles with white.
"""

import argparse
from pathlib import Path

import numpy as np
from PIL import Image

TILE = 512
STARTS = (0, 494, 988)  # 3x3 grid over 1500px with slight overlap
BLANK_FRAC = 0.25


def tile_pair(sat_path: Path, map_path: Path, out_img: Path, out_msk: Path) -> int:
    sat = np.asarray(Image.open(sat_path).convert("RGB"))
    msk = np.asarray(Image.open(map_path).convert("L"))
    if sat.shape[0] < TILE or sat.shape[1] < TILE:
        return 0
    kept = 0
    for i, y in enumerate(STARTS):
        for j, x in enumerate(STARTS):
            if y + TILE > sat.shape[0] or x + TILE > sat.shape[1]:
                continue
            s = sat[y : y + TILE, x : x + TILE]
            m = msk[y : y + TILE, x : x + TILE]
            blank = np.all(s > 250, axis=-1).mean()
            if blank > BLANK_FRAC:
                continue
            stem = f"{sat_path.stem}_{i}{j}"
            Image.fromarray(s).save(out_img / f"{stem}.png")
            Image.fromarray(((m > 127) * 255).astype(np.uint8)).save(out_msk / f"{stem}.png")
            kept += 1
    return kept


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--raw", type=Path, default=Path("data/raw"))
    p.add_argument("--out", type=Path, default=Path("data/tiles"))
    args = p.parse_args()

    for split in ("train", "valid", "test"):
        sat_dir = args.raw / split / "sat"
        map_dir = args.raw / split / "map"
        if not sat_dir.is_dir():
            continue
        out_img = args.out / split / "images"
        out_msk = args.out / split / "masks"
        out_img.mkdir(parents=True, exist_ok=True)
        out_msk.mkdir(parents=True, exist_ok=True)

        total = 0
        pairs = 0
        for sat_path in sorted(sat_dir.iterdir()):
            if sat_path.suffix.lower() not in (".tif", ".tiff", ".png"):
                continue
            candidates = [map_dir / f"{sat_path.stem}{ext}" for ext in (".tif", ".tiff", ".png")]
            map_path = next((c for c in candidates if c.exists()), None)
            if map_path is None:
                print(f"[{split}] no mask for {sat_path.name}, skipping")
                continue
            total += tile_pair(sat_path, map_path, out_img, out_msk)
            pairs += 1
        print(f"[{split}] {pairs} images -> {total} tiles")


if __name__ == "__main__":
    main()


In [ ]:
!python -m roadx.data.download --out /content/raw --all


In [ ]:
!python -m roadx.data.tile --raw /content/raw --out /content/tiles


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

tiles = Path('/content/tiles')
for split in ('train', 'valid', 'test'):
    n = len(list((tiles / split / 'images').glob('*.png')))
    print(f'{split}: {n} tiles')

sample = sorted((tiles / 'train' / 'images').glob('*.png'))[0]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(Image.open(sample)); ax[0].set_title('image'); ax[0].axis('off')
ax[1].imshow(Image.open(tiles / 'train' / 'masks' / sample.name), cmap='gray')
ax[1].set_title('mask'); ax[1].axis('off')
plt.show()


In [ ]:
!cd /content && zip -qr tiles.zip tiles
!cp /content/tiles.zip /content/drive/MyDrive/roadx/tiles.zip
!ls -lh /content/drive/MyDrive/roadx/
print('done — tiles.zip is in Drive, you can close this session')
